# Hands-On LangChain — Part 2: Document Splitting, Part 1

This maintained companion notebook uses the current LangChain text-splitter package to demonstrate character splitting, recursive splitting, overlap, separator selection, and sentence-aware splitting. The default path is deterministic and requires no API key.

## 1. Install dependencies

In [ ]:
!pip install -q "langchain-text-splitters>=0.3"

## 2. Create character-based splitters

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

chunk_size = 26
chunk_overlap = 4

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)
character_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)
print("Splitters initialized with chunk_size=26 and chunk_overlap=4.")

## 3. Compare deterministic examples

In [ ]:
text_1 = "abcdefghijklmnopqrstuvwxyz"
text_2 = "abcdefghijklmnopqrstuvwxyzabcdefg"
text_3 = "a b c d e f g h i j k l m n o p q r s t u v w x y z"

result_1 = recursive_splitter.split_text(text_1)
result_2 = recursive_splitter.split_text(text_2)
recursive_result_3 = recursive_splitter.split_text(text_3)
character_result_3 = character_splitter.split_text(text_3)

print("Recursive / 26 characters:", result_1)
print("Recursive / longer text:", result_2)
print("Recursive / spaced text:", recursive_result_3)
print("Character / spaced text:", character_result_3)

## 4. Split paragraphs without cutting across their natural boundary

In [ ]:
some_text = '''When writing documents, writers will use document structure to group content. This can convey to the reader which ideas are related. For example, closely related ideas are in sentences. Similar ideas are in paragraphs. Paragraphs form a document.

Paragraphs are typically separated by one or two carriage returns. Carriage returns are the escape sequence \n in Python. Splitting first on double newlines helps preserve the paragraph structure.'''

character_paragraph_splitter = CharacterTextSplitter(
    separator=" ", chunk_size=300, chunk_overlap=0
)
recursive_paragraph_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""], chunk_size=300, chunk_overlap=0
)

character_chunks = character_paragraph_splitter.split_text(some_text)
recursive_chunks = recursive_paragraph_splitter.split_text(some_text)
print("Character chunks:", len(character_chunks))
print("Recursive chunks:", len(recursive_chunks))
for index, chunk in enumerate(recursive_chunks, start=1):
    print(f"Recursive chunk {index}: {chunk}")

## 5. Use a regex boundary to retain sentence punctuation

In [ ]:
climate_text = '''As the world grapples with the challenges of climate change, renewable energy emerges as a beacon of hope. Solar and wind power, in particular, are transforming the energy landscape, offering sustainable alternatives to traditional fossil fuels. Governments and businesses globally are investing in clean energy initiatives to reduce carbon footprints and mitigate environmental impact. The shift towards renewables not only addresses environmental concerns but also fosters innovation, creating a brighter and more sustainable future for generations to come.'''

sentence_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=0,
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    is_separator_regex=True,
)
sentence_chunks = sentence_splitter.split_text(climate_text)
for index, chunk in enumerate(sentence_chunks, start=1):
    print(f"Sentence chunk {index}: {chunk}")

## 6. Verify the examples

In [ ]:
assert result_1 == ["abcdefghijklmnopqrstuvwxyz"]
assert result_2 == ["abcdefghijklmnopqrstuvwxyz", "wxyzabcdefg"]
assert recursive_result_3 == [
    "a b c d e f g h i j k l m",
    "l m n o p q r s t u v w x",
    "w x y z",
]
assert len(character_chunks) == 2
assert len(recursive_chunks) == 2
assert len(sentence_chunks) > 1
assert all(len(chunk) <= 150 for chunk in sentence_chunks)
assert "".join(sentence_chunks).replace(" ", "") == climate_text.replace(" ", "")
print(f"Verified {len(sentence_chunks)} sentence-aware chunks.")
print("All Part 1 splitting assertions passed.")